In [ ]:
%pip install transformers torch matplotlib

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_attentions=True)

inputs = tokenizer("""
Evaluate if the Transformer can generalize to other tasks.
""", return_tensors="pt")

# inputs = tokenizer("""
# Evaluate if the Transformer can generalize to other tasks. This task presents
# specific challenges
# """, return_tensors="pt")

# inputs = tokenizer("""
# Peter Piper picked a peck of pickled peppers a peck of pickled peppers Peter Piper picked
# """, return_tensors="pt")

# inputs = tokenizer("""
# He lives in a pineapple under the sea SpongeBob SquarePants is his name
# """, return_tensors="pt")

# inputs = tokenizer("""
# But I'm a creep
# I'm a weirdo
# What the hell am I doing here?
# I don't belong here
# """, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
seq_len = len(tokens)
special_idx = [i for i, t in enumerate(tokens) if t in ("[CLS]", "[SEP]")]

n_layers = len(outputs.attentions)
n_heads = outputs.attentions[0].shape[1]

rows = []
for layer in range(n_layers):
    for head in range(n_heads):
        A = outputs.attentions[layer][0, head].numpy()  # (seq, seq), each row sums to 1

        # 1. Peakedness: average of the single largest weight in each row (0..1, higher = sharper)
        peakedness = A.max(axis=1).mean()

        # 2. Target diversity: how many DISTINCT columns are the argmax across rows.
        #    Normalized 0..1. A sink (all point to one col) -> low; varied targets -> high.
        argmax_targets = A.argmax(axis=1)
        diversity = len(np.unique(argmax_targets)) / seq_len

        # 3. Off-diagonal mass: fraction of attention NOT on the token itself.
        off_diagonal = 1.0 - np.trace(A) / seq_len

        # Sink penalty: fraction of ALL attention mass landing on [CLS]/[SEP].
        sink_mass = A[:, special_idx].sum() / seq_len if special_idx else 0.0

        starkness = peakedness * diversity * off_diagonal * (1.0 - sink_mass)

        rows.append({
            "layer": layer, "head": head,
            "peakedness": peakedness, "diversity": diversity,
            "off_diagonal": off_diagonal, "sink_mass": sink_mass,
            "starkness": starkness,
        })

ranked = sorted(rows, key=lambda r: r["starkness"], reverse=True)

print(f"Sentence: {tokenizer.decode(inputs['input_ids'][0])}")
print(f"Scanned {n_layers} layers x {n_heads} heads = {len(rows)} combinations\n")
print(f"{'rank':>4}  {'layer':>5} {'head':>4}  {'stark':>6}  {'peak':>5}  {'divers':>6}  {'offdiag':>7}  {'sink':>5}")
for rank, r in enumerate(ranked[:10], 1):
    print(f"{rank:>4}  {r['layer']:>5} {r['head']:>4}  {r['starkness']:>6.3f}  "
          f"{r['peakedness']:>5.2f}  {r['diversity']:>6.2f}  {r['off_diagonal']:>7.2f}  {r['sink_mass']:>5.2f}")

best = ranked[0]

In [ ]:
# Render the winning head from the scan above
best_attn = outputs.attentions[best["layer"]][0, best["head"]]

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(best_attn, cmap="viridis")

ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=45, ha="right")
ax.set_yticklabels(tokens)

ax.set_xlabel("Token being attended to (key)")
ax.set_ylabel("Token doing the attending (query)")
ax.set_title(f"Starkest head: Layer {best['layer']} · Head {best['head']} "
             f"(starkness={best['starkness']:.2f})")

for i in range(best_attn.shape[0]):
    for j in range(best_attn.shape[1]):
        ax.text(j, i, f"{best_attn[i, j]:.2f}",
                ha="center", va="center",
                color="white" if best_attn[i, j] < 0.5 else "black",
                fontsize=8)

fig.colorbar(im, ax=ax, label="Attention weight")
plt.tight_layout()
plt.show()


In [ ]:
content_idx = [i for i, t in enumerate(tokens) if t not in ("[CLS]", "[SEP]")]
special_idx = [i for i, t in enumerate(tokens) if t in ("[CLS]", "[SEP]")]

# Only consider heads that keep at least this fraction of their attention on real
# words. Raise it toward 1.0 to be stricter about excluding any [CLS]/[SEP] leakage.
WORD_MASS_MIN = 0.85
# Ignore heads whose "distant" links are really just the previous/next word.
MIN_REACH = 4.0

longrange = []
for layer in range(n_layers):
    for head in range(n_heads):
        A = outputs.attentions[layer][0, head].numpy()

        # How much of this head's attention stays on real WORDS (not [CLS]/[SEP]).
        content_mass = A[np.ix_(content_idx, content_idx)].sum() / len(content_idx)

        weights, dists = [], []
        for i in content_idx:
            cands = [j for j in content_idx if abs(j - i) > 1]
            if not cands:
                continue
            # Renormalize this row over CONTENT columns only, so link_strength reflects
            # pure word-to-word attention with the [CLS]/[SEP] sink removed.
            row = A[i, content_idx]
            row = row / row.sum()
            k = max(range(len(content_idx)), key=lambda c: row[c])
            j = content_idx[k]
            if abs(j - i) <= 1:      # renormalized best is a neighbour -> not long-range
                continue
            weights.append(row[k])   # strength of the best distant word-link (word-only)
            dists.append(abs(j - i)) # how far it reaches
        if not weights:
            continue

        mean_w = np.mean(weights)    # strength of distant word links (0..1)
        mean_d = np.mean(dists)      # reach in tokens
        score = mean_w * mean_d      # strong AND far, measured on words only

        longrange.append({
            "layer": layer, "head": head,
            "mean_w": mean_w, "mean_d": mean_d,
            "content_mass": content_mass, "score": score,
        })

# Keep only heads that are word-clean AND genuinely long-range; fall back gracefully
# if the thresholds are too strict for this particular sentence.
eligible = [r for r in longrange
            if r["content_mass"] >= WORD_MASS_MIN and r["mean_d"] >= MIN_REACH]
eligible = eligible or [r for r in longrange if r["content_mass"] >= WORD_MASS_MIN]
eligible = eligible or longrange
lr_ranked = sorted(eligible, key=lambda r: r["score"], reverse=True)

print("Heads that best show LONG-RANGE, word-to-word (non-sequential) attention:")
print(f"(word_mass >= {WORD_MASS_MIN}, reach >= {MIN_REACH}; link strength is word-only)\n")
print(f"{'rank':>4}  {'layer':>5} {'head':>4}  {'score':>6}  {'link_strength':>13}  {'avg_reach':>9}  {'word_mass':>9}")
for rank, r in enumerate(lr_ranked[:8], 1):
    print(f"{rank:>4}  {r['layer']:>5} {r['head']:>4}  {r['score']:>6.2f}  "
          f"{r['mean_w']:>13.2f}  {r['mean_d']:>9.2f}  {r['content_mass']:>9.2f}")

lr = lr_ranked[0]
lr_attn = outputs.attentions[lr["layer"]][0, lr["head"]]

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(lr_attn, cmap="viridis")
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=45, ha="right")
ax.set_yticklabels(tokens)
ax.set_xlabel("Token being attended to (key)")
ax.set_ylabel("Token doing the attending (query)")
ax.set_title(f"Long-range attention: Layer {lr['layer']} · Head {lr['head']} "
             f"(avg reach = {lr['mean_d']:.1f} tokens, word_mass = {lr['content_mass']:.2f})")

for i in range(lr_attn.shape[0]):
    for j in range(lr_attn.shape[1]):
        ax.text(j, i, f"{lr_attn[i, j]:.2f}",
                ha="center", va="center",
                color="white" if lr_attn[i, j] < 0.5 else "black",
                fontsize=8)

fig.colorbar(im, ax=ax, label="Attention weight")
plt.tight_layout()
plt.show()


In [ ]:
%pip install transformer-lens

In [ ]:
from transformer_lens.model_bridge import TransformerBridge

# Wrap the SAME bert-base-uncased weights behind TransformerLens hook points.
bridge = TransformerBridge.boot_transformers(model_name, device="cpu")
bridge.enable_compatibility_mode()   # expose classic hook_z / hook_resid_post names

# Sanity check: run the tongue-twister through the bridge and list the hook points
# we'll use for activations (residual stream) and ablation (per-head output z).
_, cache = bridge.run_with_cache(inputs["input_ids"])
print("d_model:", bridge.cfg.d_model, "| n_layers:", bridge.cfg.n_layers,
      "| n_heads:", bridge.cfg.n_heads)
for name in ("blocks.0.hook_resid_pre", "blocks.0.hook_resid_post",
             "blocks.0.attn.hook_z", "blocks.0.attn.hook_pattern"):
    print(f"{name:32s} -> {tuple(cache[name].shape)}")


## From attention to activations

Everything above looked at **attention** — *how* information moves between tokens. But
attention is only the routing step. What the model actually carries forward is the
**activation**: the hidden vector it computes for every token at every layer (the
"residual stream").

Two things worth seeing:

1. **Activations grow with depth.** Each layer reads the previous activations, mixes
   them with attention, and writes an updated vector. We can watch their magnitude
   change layer by layer.
2. **Activations encode meaning**


In [ ]:
# Read activations straight from the bridge's hook-point cache -- no output_hidden_states,
# no manual plumbing. run_with_cache captures every named activation in one pass.
_, cache = bridge.run_with_cache(inputs["input_ids"])

# Residual stream: what ENTERS block 0 (embeddings) plus what LEAVES each block.
resid = [cache["blocks.0.hook_resid_pre"][0]]
resid += [cache[f"blocks.{i}.hook_resid_post"][0] for i in range(n_layers)]
act_norms = np.stack([r.norm(dim=-1).detach().numpy() for r in resid])  # (depth, seq)

# Final-layer activations, compared token-to-token by cosine similarity.
final = cache[f"blocks.{n_layers - 1}.hook_resid_post"][0]   # (seq, d_model)
final_n = final / final.norm(dim=-1, keepdim=True)
sim = (final_n @ final_n.T).detach().numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

im1 = ax1.imshow(act_norms, aspect="auto", cmap="magma")
ax1.set_xticks(range(seq_len))
ax1.set_xticklabels(tokens, rotation=45, ha="right")
ax1.set_yticks(range(len(resid)))
ax1.set_yticklabels(["embed"] + [f"L{i}" for i in range(1, len(resid))])
ax1.set_xlabel("Token")
ax1.set_ylabel("Depth (activations flow upward)")
ax1.set_title("Activation magnitude at every layer")
fig.colorbar(im1, ax=ax1, label="||activation||")

im2 = ax2.imshow(sim, cmap="viridis")
ax2.set_xticks(range(seq_len))
ax2.set_yticks(range(seq_len))
ax2.set_xticklabels(tokens, rotation=45, ha="right")
ax2.set_yticklabels(tokens)
ax2.set_title("Final activations: cosine similarity\n(repeated words -> similar vectors)")
for i in range(seq_len):
    for j in range(seq_len):
        ax2.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center",
                 color="white" if sim[i, j] < 0.6 else "black", fontsize=6)
fig.colorbar(im2, ax=ax2, label="cosine similarity")

plt.tight_layout()
plt.show()


## Ablation: does that long-range head actually matter?

We *found* an interesting long-range head by staring at attention patterns. But an
interesting-looking pattern isn't proof that the head does anything useful. The way to
test a component's role is **ablation**: knock it out and measure how much the
activations change.

With the bridge this is a one-liner intervention. Each head writes its result into the
`blocks.{i}.attn.hook_z` activation, shaped `(batch, pos, n_heads, d_head)`. We register
a `run_with_hooks` function that zeros the slice for a single head, then read the
final-layer residual stream back out and compare it to the un-ablated run. A head that
matters will visibly move the activations; a redundant head will barely move them.


In [ ]:
LAST = f"blocks.{n_layers - 1}.hook_resid_post"   # final-layer activations hook

def final_activations(layer=None, head=None):
    """Final-layer residual activations, optionally ablating one head via hook_z."""
    fwd_hooks = []
    if layer is not None:
        def ablate_z(z, hook):
            z[:, :, head, :] = 0.0          # zero this head's write (batch, pos, head, d_head)
            return z
        fwd_hooks = [(f"blocks.{layer}.attn.hook_z", ablate_z)]
    with bridge.hooks(fwd_hooks=fwd_hooks):
        _, cache = bridge.run_with_cache(inputs["input_ids"], names_filter=LAST)
    return cache[LAST][0].detach()          # (seq, d_model)

baseline = final_activations()                       # nothing removed
ablated = final_activations(lr["layer"], lr["head"]) # long-range head removed

# Per-token change in the final activation vector (L2 distance).
delta = (ablated - baseline).norm(dim=-1).numpy()

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(seq_len), delta, color="crimson")
ax.set_xticks(range(seq_len))
ax.set_xticklabels(tokens, rotation=45, ha="right")
ax.set_ylabel("Δ activation (L2)")
ax.set_title(f"Ablating Layer {lr['layer']} · Head {lr['head']}: "
             f"change in each token's final representation")
plt.tight_layout()
plt.show()

print(f"Total activation shift from removing this single head: {delta.sum():.3f}")
print("Most-affected tokens:",
      ", ".join(tokens[i] for i in np.argsort(delta)[::-1][:5]))


In [ ]:
# Ablate EVERY head one at a time and rank them by how far they move the activations.
# This is the ablation-based answer to "which heads matter?" -- compare it with the
# attention-pattern rankings we computed earlier.
importance = []
for L in range(n_layers):
    for H in range(n_heads):
        shift = (final_activations(L, H) - baseline).norm(dim=-1).sum().item()
        importance.append({"layer": L, "head": H, "shift": shift})

imp_ranked = sorted(importance, key=lambda r: r["shift"], reverse=True)

print(f"Ranked {len(imp_ranked)} heads by ablation impact (total activation shift):\n")
print(f"{'rank':>4}  {'layer':>5} {'head':>4}  {'ablation_shift':>14}")
for rank, r in enumerate(imp_ranked[:10], 1):
    tag = "  <- our long-range head" if (r["layer"], r["head"]) == (lr["layer"], lr["head"]) else ""
    print(f"{rank:>4}  {r['layer']:>5} {r['head']:>4}  {r['shift']:>14.3f}{tag}")

our_rank = next(i for i, r in enumerate(imp_ranked, 1)
                if (r["layer"], r["head"]) == (lr["layer"], lr["head"]))
print(f"\nThe long-range head (L{lr['layer']} H{lr['head']}) ranks #{our_rank} of "
      f"{len(imp_ranked)} by ablation impact.")
print("Takeaway: a striking attention *pattern* and a large ablation *effect* are")
print("related but not the same thing -- ablation is the causal test.")
